<a href="https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: Which pages in a content portfolio should a review
team prioritize first for refresh, given limited review capacity?

Decision this supports: which pages a content/SEO reviewer opens and
inspects first, out of hundreds of candidates with real search visibility.

Lane: Refresh / Content Opportunity Scoring (Lane 2).

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Starter dataset: data/raw/content_refresh_anonymized.csv (~30,000
anonymized pages, one row per content page). The full warehouse
release (FlyRank/internship-warehouse) covers dim_clients, dim_content,
and fact_content_daily_performance (78,835,655 rows, 2025-01-27 to
2026-06-30) - not used directly here, since Lane 2 permits the starter
dataset per the lane guide.

Excluded: product decision fields (health_score, priority_score,
action_type) - not shipped in this data, and would create circular
results if rebuilt and fed back in as features. No raw client names,
domains, URLs, or queries are present anywhere.

In [3]:
!git clone https://github.com/mohammadayaanahsan/flyRank-internship.git
%cd flyRank-internship

Cloning into 'flyRank-internship'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 154 (delta 67), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.89 MiB | 15.02 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/flyRank-internship/flyRank-internship


In [4]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df)}, Unique content_id: {df['content_id'].nunique()}")
print(df[["content_id","client_id","impressions_90d","trend_direction"]].head())

Rows: 30000, Unique content_id: 30000
             content_id          client_id  impressions_90d trend_direction
0  content_304f48230142  client_f369cb89fc             3803            down
1  content_a1fb4e703a9e  client_4e07408562            15320            down
2  content_9aa793d4d895  client_7f2253d7e2            12581            down
3  content_331d6c4de07b  client_19581e27de            11751          stable
4  content_d99b7a2d90ca  client_3fdba35f04            19140            down


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Unit of analysis: one content page (content_id), over a 90-day window.
Label/proxy: is_declining_label = (trend_direction == "down") - a
current-window proxy, not a future outcome.
Features: content_age_days, days_since_last_update, impressions_90d,
avg_position, ctr, word_count.
Baseline: weighted score (visibility 40%, freshness 30%, position 25%,
depth 5%) - same formula as scripts/02_baseline_score.py.
Models: Logistic Regression, Decision Tree, Random Forest.
Validation: client-grouped holdout split (GroupShuffleSplit, 25% test)
- no client's pages appear in both train and test.
Leakage checks: confirmed in work/notebooks/w06_validation_audit.ipynb.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Below: baseline vs three models on the same client-grouped split, plus
the naive-vs-grouped split comparison showing why validation design matters.

In [5]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

df2 = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")
y = (df2["trend_direction"] == "down").astype(int)
num_features = ["content_age_days","days_since_last_update","impressions_90d","avg_position","ctr","word_count"]
X = df2[num_features].replace([np.inf,-np.inf], np.nan).fillna(0)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def normalize(s):
    s = s.replace([np.inf,-np.inf], np.nan).fillna(0)
    return (s - s.min())/(s.max()-s.min()) if s.max()!=s.min() else s*0

baseline = (0.40*normalize(df2["impressions_90d"]) + 0.30*normalize(df2["days_since_last_update"])
            + 0.25*normalize(-df2["avg_position"].fillna(df2["avg_position"].max()))
            + 0.05*normalize(-df2["word_count"].fillna(df2["word_count"].max())))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=df2["client_id"]))

results = []
for k in (20,50):
    results.append(["Baseline (Week 4)", k, round(precision_at_k(baseline.iloc[te_idx].values, y.iloc[te_idx].values, k),3)])

models = {"Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
          "Decision Tree": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
          "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)}
for name, m in models.items():
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    probs = m.predict_proba(X.iloc[te_idx])[:,1]
    for k in (20,50):
        results.append([name, k, round(precision_at_k(probs, y.iloc[te_idx].values, k),3)])

results_df = pd.DataFrame(results, columns=["model","k","precision_at_k"])
print(results_df.pivot(index="model", columns="k", values="precision_at_k"))

# Naive vs grouped split (Random Forest)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_naive = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(Xr_tr, yr_tr)
naive_probs = rf_naive.predict_proba(Xr_te)[:,1]

rf_grouped = models["Random Forest"]
grouped_probs = rf_grouped.predict_proba(X.iloc[te_idx])[:,1]

for k in (20,50):
    print(f"P@{k}: naive={precision_at_k(naive_probs, yr_te.values, k):.3f}  grouped={precision_at_k(grouped_probs, y.iloc[te_idx].values, k):.3f}")

k                      20    50
model                          
Baseline (Week 4)    0.45  0.50
Decision Tree        0.60  0.60
Logistic Regression  0.65  0.66
Random Forest        0.60  0.68
P@20: naive=1.000  grouped=0.600
P@50: naive=0.940  grouped=0.680


## 5. Limitations

*What this work cannot claim.*

Small (~30K row) anonymized starter slice, not the full 79M-row
warehouse - results are directional, not production-scale. The label
is computed from the current window, not a future outcome. The naive
random split showed inflated precision (1.000/0.940) vs the honest
client-grouped split (0.600/0.680) - proof validation design matters.
No claim of predicting Google's algorithm or that refresh causes
recovery is made.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*


Output: work/outputs/final_action_queue.csv - model probability (70%)
+ baseline score (30%), with reason codes. Intended use: a reviewer
opens the top N pages matching their capacity and inspects manually -
never an automated trigger.

In [7]:
import os
os.makedirs("work/outputs", exist_ok=True)

# Rebuild the final action queue (same logic as w07)
df["model_probability"] = rf_grouped.predict_proba(X)[:, 1]

final_score = 100 * (0.70 * df["model_probability"] + 0.30 * baseline)

def reason_codes(row):
    codes = []
    if row["model_probability"] >= 0.65:
        codes.append("model_decline_risk")
    if row["model_probability"] >= 0.50 and row["impressions_90d"] >= 500:
        codes.append("visible_model_opportunity")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        codes.append("ctr_review_candidate")
    return ",".join(codes) if codes else "low_confidence"

df["final_score"] = final_score
df["reason_codes"] = df.apply(reason_codes, axis=1)

queue_rebuilt = df.sort_values("final_score", ascending=False)
queue_rebuilt.to_csv("work/outputs/final_action_queue.csv", index=False)
print("Rebuilt and saved final_action_queue.csv")

Rebuilt and saved final_action_queue.csv


In [8]:
queue = pd.read_csv("work/outputs/final_action_queue.csv")
print(queue[["content_id","final_score","reason_codes"]].head(15))

              content_id  final_score  \
0   content_72496874f806    82.014251   
1   content_4d76cdb3387b    81.203143   
2   content_4e658a59c333    81.196026   
3   content_bd823d21e6a4    81.177545   
4   content_d333007c70d9    81.169956   
5   content_9a8f08a47294    81.147898   
6   content_a10924bced3d    81.133227   
7   content_9d4e9f7b6e58    81.115121   
8   content_aa8c0ad3a60b    81.106856   
9   content_9b03dbbf0470    81.077962   
10  content_66b4046cc144    81.070774   
11  content_d6570c51c9bd    81.027692   
12  content_f30d30832c2d    81.018505   
13  content_a95a24bd6507    81.012266   
14  content_f1cdef315047    80.960441   

                                         reason_codes  
0   model_decline_risk,visible_model_opportunity,c...  
1   model_decline_risk,visible_model_opportunity,c...  
2                                  model_decline_risk  
3                                  model_decline_risk  
4                                  model_decline_risk  
5      

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper reuses these artifacts directly:
- The baseline vs. models Precision@20/@50 table (Section 4 above)
- The naive-split vs. client-grouped-split comparison table (Section 4)
  — the report's central methodology point
- The Random Forest feature importance ranking (Section 4)
- The top rows of the final ranked action queue (Section 6), with reason codes
- Chart: work/outputs/charts/top50_vs_rest.png

These come from work/notebooks/w05_model.ipynb, w06_validation_audit.ipynb,
and w07_action_playbook.ipynb, and are reproduced fresh in Section 4 and
Section 6 of this notebook so the numbers stay tied to actual code, not
just copied text.

In [10]:
import os
for f in ["work/outputs/final_action_queue.csv", "work/outputs/charts/top50_vs_rest.png"]:
    print(f, "exists:", os.path.exists(f))

work/outputs/final_action_queue.csv exists: True
work/outputs/charts/top50_vs_rest.png exists: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
